In [0]:
from pyspark.sql import functions as F

trips_df = spark.read.table("nyc_taxi.silver.yellow_trips_clean")
zones_df = spark.read.table("nyc_taxi.bronze.taxi_zones_raw")

zones_clean = zones_df.select(
    F.col("locationID").cast("integer").alias("zone_id"),
    F.col("Borough").alias("borough"),
    F.col("Zone").alias("zone")
)

daily_metrics_df = (
    trips_df
    .withColumn("trip_date",F.to_date("pickup_datetime"))
    .groupBy("trip_date")
    .agg(
        F.count("*").alias("total_trips"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("trip_distance").alias("avg_trip_distance"),
        F.avg("fare_amount").alias("avg_fare_amount"),
        F.sum("passenger_count").alias("total_passengers")
    )
    .orderBy("trip_date")
)

(
    daily_metrics_df.write.
    format("delta").
    mode("overwrite").
    saveAsTable("nyc_taxi.gold.daily_trip_metrics")
)

trips_with_zones = (
    trips_df
    .join(zones_clean,trips_df.pu_location_id == zones_clean.zone_id,"inner")
)

borough_metrics_df = (
    trips_with_zones
    .groupBy("borough",F.to_date("pickup_datetime").alias("trip_date"))
    .agg(
        F.count("*").alias("total_trips"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("trip_distance").alias("avg_trip_distance")
        )
    .orderBy(F.desc("total_revenue"))
)

(
    borough_metrics_df.write.
    format("delta").
    mode("overwrite").
    saveAsTable("nyc_taxi.gold.borough_daily_metrics")
)

print("Gold layer aggregations successfully created!")

In [0]:
%sql
SELECT count(*) FROM nyc_taxi.gold.daily_trip_metrics 
--SELECT * FROM nyc_taxi_gold.borough_daily_metrics LIMIT 10;